# Task 2.5 — SFT Re-scoring

Re-score all factors with the SFT (LoRA fine-tuned) model.

| | |
|---|---|
| **Base model** | `deepseek-ai/DeepSeek-R1-Distill-Qwen-14B` |
| **Adapter** | `models/sentiment_lora/` |
| **Input** | `../task_1/output/factors_scored/{TICKER}/*_factors.json` (~2,542 filings, ~74K factors) |
| **Output** | `output/factors_scored_sft/{TICKER}/*_factors.json` (same schema, sentiment re-scored) |

The SFT model was fine-tuned on single-factor sentiment prompts (Task 2.2). This notebook
applies that adapter to re-score every factor in the corpus, one factor at a time, using
the same prompt format the model was trained on.

## Step 1: Imports and Paths

In [ ]:
import json
import logging
import re
import sys
import time
from collections import Counter
from pathlib import Path

import torch
from peft import PeftModel
from tqdm.notebook import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# ── Paths ────────────────────────────────────────────────────────────────
_cwd = Path.cwd()
TASK1_DIR = (_cwd.parent / "task_1").resolve() if (_cwd.parent / "task_1").exists() else (_cwd / ".." / "task_1").resolve()

FACTORS_SCORED_DIR = TASK1_DIR / "output" / "factors_scored"
TICKER_MAPPING_PATH = TASK1_DIR / "ticker_mapping.json"

SFT_SCORED_DIR = _cwd / "output" / "factors_scored_sft"
LORA_DIR = _cwd / "models" / "sentiment_lora"

SFT_SCORED_DIR.mkdir(parents=True, exist_ok=True)

# ── Constants ────────────────────────────────────────────────────────────
MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"
SFT_MODEL_TAG = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B+sft_lora"
LABEL_ORDER = ["very_negative", "negative", "neutral", "positive", "very_positive"]
VALID_LABELS = set(LABEL_ORDER)
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.1

# ── Ticker -> sub-sector mapping (needed to reproduce training prompt) ───
with open(TICKER_MAPPING_PATH) as f:
    _ticker_mapping = json.load(f)
TICKER_SUBSECTOR = {}
for subsector, tickers in _ticker_mapping.items():
    for t in tickers:
        TICKER_SUBSECTOR[t] = subsector

# ── Logging ──────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
log = logging.getLogger("task_2_5")

# ── Verify paths ─────────────────────────────────────────────────────────
assert FACTORS_SCORED_DIR.exists(), f"Input directory not found: {FACTORS_SCORED_DIR}"
assert LORA_DIR.exists(), f"LoRA adapter not found: {LORA_DIR}"
print(f"Input factors:  {FACTORS_SCORED_DIR}")
print(f"Output SFT:     {SFT_SCORED_DIR}")
print(f"LoRA adapter:   {LORA_DIR}")
print(f"Ticker mapping: {len(TICKER_SUBSECTOR)} tickers across {len(_ticker_mapping)} sub-sectors")

## Step 2: Load SFT Model

Load DeepSeek-R1-Distill-Qwen-14B in 4-bit quantization with the LoRA adapter applied.
Same loading pattern as `task_2_3_evaluation.ipynb`.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Loading base model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(model, str(LORA_DIR))
model.eval()

print(f"Model loaded: {MODEL_NAME}")
print(f"LoRA adapter:  {LORA_DIR.name}")
print(f"Device:        {model.device}")
print(f"Dtype:         4-bit NF4 (compute in bfloat16)")

## Step 3: Scoring Functions

The SFT model was trained on single-factor prompts with this exact format:
- **System**: `"You are a financial analyst specializing in SEC filing analysis. Classify the sentiment of the given factor and respond with JSON only."`
- **User**: Factor details (ticker, sub-sector, category, key, summary, evidence) + classification instruction
- **Output**: `{"label": "...", "rationale": "...", "confidence": ...}`

We must reproduce this format exactly for the adapter to work correctly.

In [ ]:
# ── System prompt (matches SFT training format exactly) ──────────────────
SFT_SYSTEM_PROMPT = (
    "You are a financial analyst specializing in SEC filing analysis. "
    "Classify the sentiment of the given factor and respond with JSON only."
)


def format_evidence_text(evidence_list: list[dict]) -> str:
    """Flatten evidence array into a single text string.

    Reproduces the same format used when building the SFT training dataset
    (task_2_1_annotation_dataset.ipynb, Cell 5).
    """
    parts = []
    for ev in evidence_list:
        text = ev.get("text", "").strip()
        if text:
            section = ev.get("section", "")
            subsection = ev.get("subsection", "")
            source = f" [{section}/{subsection}]" if section else ""
            parts.append(f"{text}{source}")
    return " | ".join(parts) if parts else ""


def build_user_prompt(
    factor: dict,
    ticker: str,
    form: str,
    sub_sector: str,
) -> str:
    """Build the user prompt for a single factor.

    Matches the format from task_2_1_annotation_dataset.ipynb (format_user_prompt).
    """
    evidence = format_evidence_text(factor.get("evidence", []))
    # Truncate evidence if too long (same 1500 char limit as training)
    if len(evidence) > 1500:
        evidence = evidence[:1500] + "..."

    return (
        f"Given the following factor extracted from a {form} filing "
        f"for {ticker} ({sub_sector} sub-sector), classify the sentiment.\n\n"
        f"Category: {factor['category']}\n"
        f"Factor: {factor['key']}\n"
        f"Summary: {factor['summary']}\n"
        f"Evidence: {evidence}\n\n"
        f"Classify the sentiment as one of: very_negative, negative, neutral, positive, very_positive.\n"
        f"Provide a brief rationale and confidence score (0.0-1.0).\n"
        f"Respond with JSON only."
    )


def parse_json_response(text: str) -> dict | None:
    """Parse JSON from model output, handling think blocks and markdown fences."""
    # Strip <think>...</think> block if present
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

    # Strip markdown code fences
    text = re.sub(r"^```(?:json)?\s*\n?", "", text, flags=re.MULTILINE)
    text = re.sub(r"\n?```\s*$", "", text, flags=re.MULTILINE)
    text = text.strip()

    # Try direct parse
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Fallback: find first { to last }
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        try:
            return json.loads(text[start : end + 1])
        except json.JSONDecodeError:
            pass

    return None


def score_single_factor(
    model,
    tokenizer,
    factor: dict,
    ticker: str,
    form: str,
    sub_sector: str,
) -> dict | None:
    """Run SFT inference on a single factor.

    Returns {"label": ..., "rationale": ..., "confidence": ...} or None on failure.
    """
    user_prompt = build_user_prompt(factor, ticker, form, sub_sector)

    messages = [
        {"role": "system", "content": SFT_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

    # Build input with empty think block to skip reasoning (same as evaluation)
    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    input_text += "<think>\n</think>\n"

    inputs = tokenizer(
        input_text, return_tensors="pt", add_special_tokens=False
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )

    # Decode only generated tokens
    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1] :],
        skip_special_tokens=True,
    )

    parsed = parse_json_response(generated)
    if parsed is None:
        return None

    label = (parsed.get("label") or parsed.get("sentiment", "")).strip().lower()
    if label not in VALID_LABELS:
        return None

    return {
        "label": label,
        "rationale": parsed.get("rationale", ""),
        "confidence": min(1.0, max(0.0, float(parsed.get("confidence", 0.5)))),
    }


def process_filing_sft(
    factor_file: Path,
    model,
    tokenizer,
) -> dict | None:
    """Score all factors in one filing with the SFT model.

    Returns the updated factor JSON with SFT sentiment, or None on failure.
    """
    with open(factor_file) as f:
        data = json.load(f)

    factors = data.get("factors", [])
    if not factors:
        return None

    ticker = data.get("ticker", factor_file.parent.name)
    form = data.get("form", "")
    sub_sector = TICKER_SUBSECTOR.get(ticker, "unknown")

    scored_count = 0
    parse_errors = 0

    for factor in factors:
        result = score_single_factor(model, tokenizer, factor, ticker, form, sub_sector)
        if result is not None:
            factor["sentiment"] = result
            scored_count += 1
        else:
            # Keep existing sentiment as fallback, mark as failed
            factor["sentiment"] = None
            parse_errors += 1

    # Update model tag
    data["model"] = SFT_MODEL_TAG

    return data


print("Scoring functions defined:")
print("  - format_evidence_text(evidence_list)")
print("  - build_user_prompt(factor, ticker, form, sub_sector)")
print("  - parse_json_response(text)")
print("  - score_single_factor(model, tokenizer, factor, ticker, form, sub_sector)")
print("  - process_filing_sft(factor_file, model, tokenizer)")

## Step 4: Discover Factor Files

Glob all factor files from task_1 output. Resume-safe: skip any filing whose
output already exists in `factors_scored_sft/`.

In [ ]:
# ── Discover all factor files ────────────────────────────────────────────
all_factor_files = sorted(FACTORS_SCORED_DIR.rglob("*_factors.json"))
print(f"Total factor files found: {len(all_factor_files)}")

# ── Count total factors ─────────────────────────────────────────────────
total_factors = 0
for f in all_factor_files:
    with open(f) as fh:
        data = json.load(fh)
    total_factors += len(data.get("factors", []))
print(f"Total factors across all files: {total_factors:,}")

# ── Filter out already-processed (resume-safe) ──────────────────────────
pending_files = []
already_done = 0

for factor_file in all_factor_files:
    ticker = factor_file.parent.name
    out_file = SFT_SCORED_DIR / ticker / factor_file.name
    if out_file.exists():
        already_done += 1
    else:
        pending_files.append(factor_file)

print(f"\nAlready processed: {already_done}")
print(f"Pending:           {len(pending_files)}")

# ── Show ticker breakdown of pending ─────────────────────────────────────
if pending_files:
    pending_tickers = Counter(f.parent.name for f in pending_files)
    print(f"Pending tickers:   {len(pending_tickers)}")
    print(f"\nTop 10 tickers by pending filings:")
    for ticker, count in pending_tickers.most_common(10):
        print(f"  {ticker:6s}: {count} filings")

## Step 5: Run SFT Scoring

Process each filing one at a time. Within each filing, score each factor individually.
Save output after each filing for resume-safety. Free GPU cache periodically.

In [ ]:
total_filings_processed = 0
total_factors_scored = 0
total_factors_failed = 0
total_filings_failed = 0

t0 = time.time()

for i, factor_file in enumerate(tqdm(pending_files, desc="SFT rescoring")):
    ticker = factor_file.parent.name
    out_dir = SFT_SCORED_DIR / ticker
    out_dir.mkdir(parents=True, exist_ok=True)
    out_file = out_dir / factor_file.name

    # Double-check resume safety (in case of concurrent runs)
    if out_file.exists():
        continue

    try:
        result = process_filing_sft(factor_file, model, tokenizer)

        if result is not None:
            # Count results
            n_scored = sum(1 for f in result["factors"] if f.get("sentiment") is not None)
            n_failed = len(result["factors"]) - n_scored

            # Save immediately
            with open(out_file, "w") as f:
                json.dump(result, f, indent=2)

            total_filings_processed += 1
            total_factors_scored += n_scored
            total_factors_failed += n_failed
        else:
            total_filings_failed += 1
            log.warning(f"  {factor_file.name}: no factors, skipped")

    except Exception as e:
        total_filings_failed += 1
        log.error(f"  {factor_file.name}: CRASHED — {e}")

    # Free GPU cache every 10 filings
    if (i + 1) % 10 == 0:
        torch.cuda.empty_cache()

    # Progress stats every 50 filings
    if (i + 1) % 50 == 0:
        elapsed = time.time() - t0
        rate = (i + 1) / elapsed * 60  # filings per minute
        remaining = (len(pending_files) - i - 1) / (rate / 60) if rate > 0 else 0
        log.info(
            f"  Progress: {i+1}/{len(pending_files)} filings | "
            f"{total_factors_scored:,} factors scored | "
            f"{rate:.1f} filings/min | "
            f"ETA: {remaining/60:.1f} hrs"
        )

elapsed = time.time() - t0
print(f"\n{'='*60}")
print(f"SFT RESCORING COMPLETE")
print(f"{'='*60}")
print(f"Filings processed: {total_filings_processed}")
print(f"Filings failed:    {total_filings_failed}")
print(f"Factors scored:    {total_factors_scored:,}")
print(f"Factors failed:    {total_factors_failed:,}")
print(f"Parse success:     {total_factors_scored / max(1, total_factors_scored + total_factors_failed) * 100:.1f}%")
print(f"Total time:        {elapsed/3600:.2f} hrs ({elapsed/60:.1f} min)")
print(f"Rate:              {total_factors_scored / max(1, elapsed) * 60:.1f} factors/min")

## Step 6: Validation — Label Distribution Comparison

Compare the label distributions between the base model (task_1 scored) and the SFT model
to verify the re-scoring produced reasonable results.

In [ ]:
def collect_labels(base_dir: Path) -> Counter:
    """Collect sentiment label counts from all factor files in a directory."""
    counts = Counter()
    null_count = 0
    for fpath in sorted(base_dir.rglob("*_factors.json")):
        with open(fpath) as f:
            data = json.load(f)
        for fac in data.get("factors", []):
            sentiment = fac.get("sentiment")
            if isinstance(sentiment, dict) and sentiment.get("label"):
                counts[sentiment["label"]] += 1
            else:
                null_count += 1
    return counts, null_count


# ── Base model labels ────────────────────────────────────────────────────
print("Collecting base model labels...")
base_counts, base_nulls = collect_labels(FACTORS_SCORED_DIR)

# ── SFT model labels ────────────────────────────────────────────────────
print("Collecting SFT model labels...")
sft_counts, sft_nulls = collect_labels(SFT_SCORED_DIR)

# ── Print comparison ─────────────────────────────────────────────────────
base_total = sum(base_counts.values())
sft_total = sum(sft_counts.values())

print(f"\n{'='*60}")
print(f"LABEL DISTRIBUTION COMPARISON")
print(f"{'='*60}")
print(f"{'Label':<16s} | {'Base':>8s} {'%':>6s} | {'SFT':>8s} {'%':>6s} | {'Delta':>6s}")
print(f"{'-'*16}-+-{'-'*15}-+-{'-'*15}-+-{'-'*6}")

for label in LABEL_ORDER:
    bc = base_counts.get(label, 0)
    sc = sft_counts.get(label, 0)
    bp = bc / max(1, base_total) * 100
    sp = sc / max(1, sft_total) * 100
    delta = sp - bp
    print(f"{label:<16s} | {bc:>8,d} {bp:>5.1f}% | {sc:>8,d} {sp:>5.1f}% | {delta:>+5.1f}%")

print(f"{'-'*16}-+-{'-'*15}-+-{'-'*15}-+-{'-'*6}")
print(f"{'TOTAL':<16s} | {base_total:>8,d}       | {sft_total:>8,d}       |")
print(f"{'NULL/failed':<16s} | {base_nulls:>8,d}       | {sft_nulls:>8,d}       |")

# ── Quick sanity check on a few samples ──────────────────────────────────
print(f"\n{'='*60}")
print("SAMPLE COMPARISON (first 5 factors from first SFT file)")
print(f"{'='*60}")

sft_files = sorted(SFT_SCORED_DIR.rglob("*_factors.json"))
if sft_files:
    sample_sft = sft_files[0]
    # Find the corresponding base file
    ticker = sample_sft.parent.name
    sample_base = FACTORS_SCORED_DIR / ticker / sample_sft.name

    with open(sample_sft) as f:
        sft_data = json.load(f)
    with open(sample_base) as f:
        base_data = json.load(f)

    print(f"File: {sample_sft.name}")
    print(f"Ticker: {sft_data['ticker']}, Form: {sft_data['form']}, Date: {sft_data['filing_date']}")
    print()

    for i, (bf, sf) in enumerate(zip(base_data["factors"][:5], sft_data["factors"][:5])):
        base_label = bf.get("sentiment", {}).get("label", "N/A") if isinstance(bf.get("sentiment"), dict) else "N/A"
        sft_label = sf.get("sentiment", {}).get("label", "N/A") if isinstance(sf.get("sentiment"), dict) else "N/A"
        match = "==" if base_label == sft_label else "!="
        print(f"  [{i}] {bf['key']:<30s}  base={base_label:<14s} sft={sft_label:<14s} {match}")
else:
    print("No SFT output files found yet.")